# Imports and data:

In [1]:
import pandas as pd
import numpy as np
import regex as re

In [2]:
w1 = pd.read_excel("data/WINNERS 2000-2010 - 05.xlsx")
w2 = pd.read_excel("data/WINNERS 2011-2020 - 05.xlsx")
w3 = pd.read_excel("data/WINNERS 2021-2025 - 05.xlsx")
w4 = pd.read_excel("data/WINNERS 2026-2030 - 05.xlsx")

In [3]:
winners = pd.concat([w1, w2, w3, w4], ignore_index=True)

winners.shape

(43457, 10)

In [4]:
winners.dtypes

Name                   object
School                 object
Year                    int64
City                   object
Award                  object
Division               object
Category               object
School / Company       object
Division / Position    object
Dance Category         object
dtype: object

In [5]:
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [6]:
winners.isna().sum()

Name                     554
School                 36132
Year                       0
City                       0
Award                    174
Division               36913
Category               36477
School / Company        8286
Division / Position    11327
Dance Category         10780
dtype: int64

- Some awards are for the school overall rather than a person
- Some awards are for the name of the dance rather than people
- School awards wouldn't have a Dance Category or Position, which would explain some entries being null
- NaNs in certain columns can actually tell you what kind of award is being given:
    - NaN in School/Company ; School Award
- Some people don't belong in a School/Company

# Data Cleanup:

- Whenever Name and School have NaNs, there are no rows where School/Company is empty
    - So: replace School with whatever is in the School/Company for that row

In [7]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
7717,NaN,NaN,2011,"Torrington, CT",Outstanding School Award,NaN,NaN,"Washington School of Ballet, DC, USA",NaN,Special Awards
7806,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,The Art of Classical Ballet,NaN,Special Awards
7807,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,America's Ballet School,NaN,Special Awards
7905,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Los Angeles Ballet Academy, CA",NaN,Special Awards
7906,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Westlake School for the Performing Arts, CA",NaN,Special Awards
...,...,...,...,...,...,...,...,...,...,...
42644,NaN,NaN,2026,"Boca Raton, FL",Outstanding School Award,NaN,NaN,"Stars Dance Studio, FL",Special Awards,NaN
42928,NaN,NaN,2026,"Indianapolis, IN",Outstanding School Award,NaN,NaN,"Indiana Ballet Conservatory, IN",Special Awards,NaN
43200,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"N&D Ballet, MA",Special Awards,NaN
43201,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"Koltun Ballet Boston, MA",Special Awards,NaN


In [8]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna()) &
    (winners["School / Company"].isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category


In [9]:
mask = (winners["Name"].isna()) & (winners["School"].isna())

winners.loc[mask, "School"] = winners.loc[mask, "School / Company"]
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [10]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category


- Inconsistent patterns for recording names:
    - First Name Last Name(s)
    - Last Name, First Name
    - First Name Last Name, [US state] (sometimes followed by what US state they're from) 
    - First Name Middle Name(s) and Last Name(s)
    - [1st person] & [2nd person]
    - [1st person], [2nd person]
    - [1st person] (age), [2nd person] (age)
    - (age)
    - [1st person] age, [2nd person] age
    - First Name Last Name (age), (dance that the person/people performed)
    - dance that the person/people performed, ([1st person], [2nd person]), 
- Removing hanging whitespace from all string columns since it will effect analysis
- Turn everything uppercase, since there are inconsistent capitalization (or lack thereof)

In [11]:
winners = winners.apply(lambda col: col.str.upper() if col.dtype == "object" else col)

In [12]:
def split_2_names(string):
    if pd.isna(string):
        return string  # keep NaN
    
    if isinstance(string, str):
        return [s.strip() for s in string.split("&")]
    
    return string

winners.Name = winners.Name.apply(split_2_names)
winners = winners.explode("Name", ignore_index=True)

In [13]:
winners["Name"] = winners.Name.str.strip()

In [14]:
names = sorted(winners.Name.dropna().unique())

with open("data/yagp_names.txt", "w") as file:
    file.writelines(
        name + "\n" for name in names
    )

In [ ]:
winners[
    winners.Name.str.contains(r"\(\d{2}\)", regex=True, na=False)
].Name.unique()

array(['MARIA CLARA COELHO (13), DAVI CHAGAS (16)',
       'BRIANNA BERRIOS (13), BLAKE KESSLER (14)',
       'GAYEON JUNG, (19), KIMIN KIM (19)',
       'COURTNEY GUNSTEEN (13), TANNER BLECK (14)',
       'SERENA SOVDSNES (15), JULIO MEDEROS (24)',
       'NICOLE FANNEY (13), PRESTON CHAMBLEE (17)',
       'BELLA KIRBY (11), AUSTEN ACEVEDO (13)',
       'GABRIELLA STILO (13), BRANDON CARPIO (15)',
       'VICTORIA WONG (15), CHANDLER PROCTOR (17)',
       'VICTORIA WONG(15), CHANDLER PROCTOR (17)',
       'SEUNG YEONG YANG (15), SUNWOO LEE (17)',
       'CLAIRE RATHRUN (18), RAMMARU SHINDO (21)',
       'GABRIELLA STILO (14), FRANCISCO SERRANO (14)',
       'WENYU GUO (19), STEFAN GONCALVEZ (19)',
       'AO WANG (17), LANG MA (19)',
       'MO CEN ZHAO (15) ZACHAY DOWNER (17)',
       'JESSICA HE (16), ANDRES ZUNIGA (16)',
       'TATE RAE (11), LAZARO MORALES (12)',
       'MAGGIE CHADBOURNE (13), ROBERTO BARQUI (16)',
       'MADELEINE GARDELLA (16), PETER WEIL (18)',
       'PENEL

In [28]:
winners[
    winners.Name.str.contains(r",", regex=True, na=False)
].Name.unique()

array(['KORY GLATMAN, NJ', 'MELISSA LORRY, MA',
       'ARIZONA BALLET SCHOOL, AZ', 'APRIL RAE, MD',
       'BILLS, BILLS, BILLS', 'RITA STROM, CA', 'SACHA DE SOLA, FL',
       'MEGAN CARLO, FL', 'BROTHER, CAN YOU SPARE A DIME',
       'ADAMS SCHOOL OF DANCE, VT', 'PAS DE DEUX, "GISELLE", ACT III',
       'ODALISQUES, LE CORSAIRE', 'GRANDE PAS DE DEUX, LE CORSAIRE',
       'ENGLISH STRING SUITE, 4TH', 'ODALISQUE, LE CORSAIRE',
       'CONCERTO GROSSO, 4TH MOVEMENT', 'SWAN LAKE, PAS DE TROIS',
       'LA FILLE MAL GARDEE, THE CLOGS DANCE', 'MIRTH, FIRST MOVEMENT',
       'WHEAT PAS DE DEUX, COPPELIA', 'SODA BALLET ACADEMY, JAPAN',
       'GRAND PAS CLASSIQUE, PAS DE DEUX', 'ROCHESTER, NY',
       'PERHAPS, PERHAPS, PERHAPS', 'KENNETH DUANE SHELBY, JR',
       'COPPELIA, THE DOLL', 'KENNETH SHELBY, JR', 'KINSEY NOVAK, 13',
       'MASUMI, YOSHIMOTO', 'ALEXANDRIA, MARX', 'CAROLIE, LAUVETZ',
       'IMPACT DANCE OF ATLANTA, GA',
       'MORNING STAR DANCE ACADEMY OF ATLANTA, GA',
       'I

## Separating awards for schools and people:
- Not all schools are in the official YAGP partner school list (International, US, Affiliate Summer Programs)

In [16]:
# school_awards = winners[
#     (winners.Name.isna()) & 
#     (winners.School.isna() == False) &
#     (winners.Award.isna() == False)
# ]

# school_awards

In [17]:
# dancers = winners[
#     (winners.Name.isna() == False) &
#     (winners.Name.isna() == False) & 
# ]

# Interesting Aggregates:

In [18]:
winners.Name.value_counts()

Name
FACULTY                         43
CAITLYN DIONEDA                 36
KATIA ALMAYEVA                  33
PAS DE DEUX FROM LE CORSAIRE    31
AURORA KELLOGG                  30
                                ..
DARK ROAD                        1
RUNNING                          1
FLIPBOOK                         1
RHYTHMS OF TEA                   1
DARLI NOGARR                     1
Name: count, Length: 16690, dtype: int64

In [19]:
winners.School.value_counts()

School
SOUTHLAND BALLET ACADEMY                   134
BALLET ACADEMY OF TEXAS, TX, USA           113
INTERNATIONAL BALLET SCHOOL, CO             92
THE ROCK SCHOOL FOR DANCE EDUCATION, PA     89
ORLANDO BALLET SCHOOL                       89
                                          ... 
DARIA BEARDEN SCHOOL OF DANCE                1
SADDLEBACK DANCE CENTER, CA                  1
DEANE DANCE CENTER, CA                       1
PENNSYLVANIA BALLET CONSERVATORY, PA         1
DANCE COLLECTIVE, CANADA                     1
Name: count, Length: 1952, dtype: int64

In [20]:
winners.groupby(["School", "Award"])["Year"].count()

School                                        Award                   
1 DANCE ACADEMY                               TOP 12                       1
A & A BALLET, IL                              OUTSTANDING SCHOOL AWARD     1
A TOUCH OF CLASS PERFORMING ARTS.             TOP 12                       1
A&A BALLET, IL                                OUTSTANDING SCHOOL AWARD     2
ACADEMIA ANNARELLA                            OUTSTANDING SCHOOL AWARD     1
                                                                          ..
ZAMUEL BALLET SCHOOL, CO, USA                 TOP 12                      39
                                              TOP 6                        1
ZEAL STUDIOS                                  TOP 12                       1
                                              TOP 14                       1
ÉCOLE SUPÉRIEURE DE BALLET DU QUÉBEC, CANADA  OUTSTANDING SCHOOL AWARD     1
Name: Year, Length: 3604, dtype: int64

In [21]:
winners.groupby(["Year", "School", "Award"])[["Year"]].count()

Year
Year School                                      Award                                
2000 ACADEMY OF DANCE ARTS, IL, USA              1ST PLACE                           3
                                                 2ND PLACE                           1
                                                 3RD PLACE (TIE)                     2
                                                 BRONZE MEDAL (TIE)                  1
                                                 OUTSTANDING CHOREOGRAPHER AWARD     1
...                                                                                ...
2026 THE SCHOOL OF CADENCE BALLET, ON            OUTSTANDING SCHOOL AWARD            1
     TRADITIONAL DANCE CATEGORY                  TOP 12                              1
     TURNING POINTE DANCE ACADEMY, MD            OUTSTANDING SCHOOL AWARD            1
     VITACCA SCHOOL FOR DANCE, TX                OUTSTANDING SCHOOL AWARD            1
     WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA OUTSTANDING SCHOOL AWARD            1

[4543 rows x 1 columns]